# Module 3 — Pitch Tunneling

How well does a pitcher share a tunnel window at the hitter decision point (23 ft from plate) while delivering pitches that arrive at different plate locations?

**tunnel_ratio = plate_divergence / (tunnel_distance + 0.01)**  
Higher = same look, different destination = good tunnel.

In [ ]:
import sys
sys.path.insert(0, "../modules")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from ingest import pull_statcast_range
from tunneling import (
    estimate_tunnel_location,
    build_pitch_pairs,
    build_tunnel_profile,
)
%matplotlib inline

In [ ]:
df = pull_statcast_range("2023-06-01", "2023-06-07", label="test")
df_t = estimate_tunnel_location(df)
print(f"{len(df_t):,} pitches, tunnel coords computed")

In [ ]:
pairs = build_pitch_pairs(df_t)
print(f"{len(pairs):,} cross-type pairs")
print(pairs[["pitcher","pitch_type_a","pitch_type_b","tunnel_distance","plate_divergence","tunnel_ratio"]].head(10))

In [ ]:
# Distribution of tunnel ratio by pitch-family pair
for (fa, fb), grp in pairs.groupby(["family_a","family_b"]):
    if len(grp) > 100:
        print(f"{fa:10s} -> {fb:10s}: n={len(grp):4d}  mean_ratio={grp["tunnel_ratio"].mean():.2f}  mean_dist={grp["tunnel_distance"].mean():.3f} ft")

In [ ]:
profile = build_tunnel_profile(df)
print(f"{len(profile)} pitchers scored")
profile.nlargest(10, "tunnel_score_pct")[["pitcher","pair_count","tunnel_distance_mean","plate_divergence_mean","tunnel_score_weighted","tunnel_score_pct"]]

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(pairs["tunnel_distance"], bins=50, color="steelblue", edgecolor="white")
ax1.set_xlabel("Tunnel Distance (ft)")
ax1.set_title("Separation at Decision Point (23 ft)")
ax2.hist(pairs["tunnel_ratio"].clip(0, 40), bins=50, color="coral", edgecolor="white")
ax2.set_xlabel("Tunnel Ratio")
ax2.set_title("Plate Divergence / Tunnel Distance")
plt.tight_layout()
plt.show()